# Optuna: LightGBM
**tabular › tuning_regression** · notebook `lightgbm_optuna`

**Kapan dipakai:** Tuning pertama yang sebaiknya dicoba: LightGBM cepat, ruang pencariannya jelas. Cocok bila waktu lomba terbatas.

**Jebakan umum:** Tuning menaikkan skor CV tapi bisa overfit ke fold (selisih CV vs valid membesar). Jaga jumlah trial wajar, dan nilai ulang pada validation yang tidak disentuh.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "lightgbm_optuna"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "tuning_regression", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Ruang pencarian
Fungsi `make_model(trial)` mendefinisikan hiperparameter yang dicari dan rentangnya.

In [ ]:
from lightgbm import LGBMRegressor

def make_model(trial):
    return LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 128, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 60),
        subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        random_state=42, n_jobs=-1, verbose=-1)

## 5. Jalankan pencarian (Optuna, TPE)
Objective = rata-rata skor CV pada data **training**. Naikkan `N_TRIALS`/`TIMEOUT` saat lomba.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS, TIMEOUT = 15, 180          # demo cepat; saat lomba mis. 100 trial / 900 detik

def objective(trial):
    pipe = build_pipeline(make_model(trial), NUM, CAT, kind=KIND)
    _, s = cv_score(pipe, Xtr, ytr, TASK, SCORING, N_SPLITS, SEED, groups=GTR)
    return s["valid_mean"]            # sklearn scoring: lebih besar = lebih baik, jadi maximize

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT)
print("trial selesai:", len(study.trials))
print("skor CV terbaik:", round(study.best_value, 4))
print("param terbaik:", study.best_params)

## 6. Lihat jalannya pencarian
Grafik kiri: apakah masih membaik atau sudah datar. Grafik kanan: hiperparameter mana yang paling berpengaruh.

In [ ]:
vals = pd.Series([t.value for t in study.trials])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(vals, ".", alpha=.5, label="trial"); ax[0].plot(vals.cummax(), "r-", label="terbaik sejauh ini")
ax[0].set(xlabel="trial", ylabel=SCORING, title="Riwayat tuning"); ax[0].legend()
try:
    imp = optuna.importance.get_param_importances(study)
    pd.Series(imp).sort_values().plot.barh(ax=ax[1], title="Pengaruh hiperparameter")
except Exception as ex:
    ax[1].set_title(f"importance tidak tersedia: {type(ex).__name__}")
fig.tight_layout(); fig.savefig(OUT / "tuning.png", dpi=130); plt.show()

## 7. Uji hasil pada validation
Selisih besar antara skor CV terbaik dan validation = tanda overfit ke fold. Validation ini tidak dipakai saat mencari.

In [ ]:
pipe = build_pipeline(make_model(optuna.trial.FixedTrial(study.best_params)), NUM, CAT, kind=KIND).fit(Xtr, ytr)
print(f"CV terbaik (train): {study.best_value:.4f}")

In [ ]:
pred = pipe.predict(Xva)
metrics = regression_metrics(yva, pred)
display(show_table(metrics))
plot_regression(yva, pred, path=OUT / "regression.png"); plt.show()

In [ ]:
save_json({"best_cv": study.best_value, "best_params": study.best_params, "valid": metrics, "n_trials": len(study.trials)}, OUT / "metrics.json")

## Latih ulang pada semua data → submission → simpan

In [ ]:
final = build_pipeline(make_model(optuna.trial.FixedTrial(study.best_params)), NUM, CAT, kind=KIND).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols)
    Xte = Xte.reindex(columns=X.columns)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, final.predict(Xte), ID_COL or "id", TARGET, OUT / "submission.csv")
    display(sub.head(3))
save_model({"pipe": final, "cols": cols}, OUT / "model.joblib")
reload_pipe = load_model(OUT / "model.joblib")["pipe"]
assert np.allclose(reload_pipe.predict(X.head(50)), final.predict(X.head(50))), "model muat-ulang beda"
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan dengan model tanpa tuning di `notebooks/tabular/<task>/`. Kalau naiknya kecil, waktu lebih baik dipakai untuk **feature engineering** atau **ensemble**.
- Tuning tidak memperbaiki kebocoran data atau fitur yang buruk.